# Unit 10: Customer Churn Prediction
## Logistic Regression Model for Telecommunications Customer Retention
### IN503 - Introduction to Machine Learning

**Student:** Danielle Holmes
**Date:** August 31, 2026
**Course:** IN503 - Introduction to Machine Learning

---

## Project Overview

Customer churn is expensive. Acquiring a replacement customer costs a telecommunications company substantially more than retaining an existing one, so the ability to identify who is likely to leave before they do determines whether a retention budget is spent usefully or wasted.

This project builds a logistic regression model on records for 7,043 customers, covering demographics, subscribed services, and account details. The target is whether a customer left the company. The work runs the full sequence from exploration through preprocessing, tuning, evaluation, coefficient interpretation, and a saved model with a working prediction function.

The goal is not only prediction. A model that scores well but explains nothing gives a retention team no direction, so this notebook treats the coefficients as a primary deliverable alongside the accuracy figures.

## Phase 1: Environment Setup

In [ ]:
!pip install kagglehub

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             precision_score, recall_score, f1_score,
                             roc_curve, auc, roc_auc_score, ConfusionMatrixDisplay)
import joblib

# Set pandas display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Libraries imported successfully.")

## Phase 2: Data Acquisition and Exploration

In [ ]:
# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")
filepath = path + '/WA_Fn-UseC_-Telco-Customer-Churn.csv'
print("Path to dataset files: ", filepath)

In [ ]:
# Load the dataset
df = pd.read_csv(filepath)

# Display the first 5 rows
print("First 5 rows of the dataset:")
print(df.head())

In [ ]:
print("\nShape of the dataset: ")
print(df.shape)

print("\nDataFrame Info: ")
df.info()

In [ ]:
print("\nDescriptive statistics for numerical features:")
print(df.describe())

In [ ]:
print("\nMissing values before handling: ")
print(df.isnull().sum())

# Check for empty strings in TotalCharges
total_charges_missing = df[df['TotalCharges'] == ' '].shape[0]
print(f"\nNumber of empty strings in 'TotalCharges': {total_charges_missing}")

In [ ]:
# Replace empty strings with NaN and convert values to float
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan).astype(float)

# Re-check for missing values
print("\nMissing values after handling 'TotalCharges' type: ")
print(df.isnull().sum())

In [ ]:
# Inspect the rows that are missing TotalCharges before removing them
print("Tenure values for rows missing TotalCharges:")
print(df[df['TotalCharges'].isna()]['tenure'].value_counts())

In [ ]:
# Drop rows with missing values (small number - 11 rows)
df.dropna(inplace=True)

# Verify cleanup
print("\nMissing values after dropping rows: ")
print(df.isnull().sum())
print(f"New dataset shape: {df.shape}")

### Data Quality Findings

The most consequential problem in this dataset is invisible to a standard missing value check. `TotalCharges` loads as an object rather than a numeric column because a small number of rows contain a single space instead of a value, and `isnull()` counts a space as present data. Converting those spaces to `NaN` and casting the column to float exposes them.

The check above reveals what those rows represent. Every record missing `TotalCharges` has a tenure of zero, meaning these are brand new customers who have not yet been billed. The value is not missing through error, it is missing because nothing has been charged yet. Removing eleven records out of 7,043 is a reasonable choice at this scale, though it is worth recognizing that this quietly excludes the newest customers from the analysis, and new customers are precisely the group most at risk of leaving.

In [ ]:
print("\nChurn distribution:")
print(df['Churn'].value_counts())

# Calculate percentages
churn_percentage = df['Churn'].value_counts(normalize=True) * 100
print("\nChurn percentage:")
print(churn_percentage)

### Target Variable Findings

The classes are imbalanced, with substantially more retained customers than churned ones. That imbalance sets the standard against which any accuracy figure has to be judged, because a model that predicted "no churn" for every customer would already score the majority class percentage while identifying not a single person at risk of leaving.

This is why the evaluation below reports precision, recall, and AUC alongside accuracy. For a retention use case, recall on the churn class is the metric that matters most, since a customer the model fails to flag is a customer nobody attempts to save.

### Exploratory Visualizations

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(data=df, x='Contract', hue='Churn')
plt.title('Churn Count by Contract Type')
plt.xlabel('Contract Type')
plt.ylabel('Number of Customers')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(data=df, x='InternetService', hue='Churn')
plt.title('Churn Count by Internet Service')
plt.xlabel('Internet Service')
plt.ylabel('Number of Customers')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='MonthlyCharges', hue='Churn', multiple='stack', kde=True)
plt.title('Distribution of Monthly Charges by Churn Status')
plt.xlabel('Monthly Charges')
plt.ylabel('Count')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='tenure', bins=30, kde=True, hue='Churn', multiple='stack')
plt.title('Distribution of Tenure by Churn Status')
plt.xlabel('Tenure (Months)')
plt.ylabel('Count')
plt.show()

In [ ]:
# Create temporary encoded churn variable
df['Churn_encoded'] = df['Churn'].map({'No': 0, 'Yes': 1})
numerical_df = df[['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn_encoded']]

plt.figure(figsize=(10, 8))
sns.heatmap(numerical_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap of Numerical Features')
plt.show()

# Clean up temporary column
df.drop('Churn_encoded', axis=1, inplace=True)

### Business Insights from the Visualizations

**Contract type is the clearest signal.** Month-to-month customers churn at a far higher rate than those on one or two year agreements. The reading is straightforward, since a month-to-month customer faces no exit cost and can leave whenever a competitor makes a better offer, while a contracted customer has a reason to stay through a period of dissatisfaction. This points to the single most actionable lever available: moving customers onto longer agreements.

**Fiber optic customers churn more than DSL customers.** This is counterintuitive, because fiber is the premium product. The likely explanations are price, since fiber costs more, or unmet expectations around service reliability. Either way it deserves investigation, because the company is losing its highest-paying subscribers.

**Higher monthly charges track with churn.** The stacked distribution shows churned customers concentrated in the upper charge ranges. This overlaps with the fiber finding, since fiber subscribers pay more, and it raises the question of whether customers perceive the price as matching the value delivered.

**Tenure is strongly protective.** Churn concentrates heavily in the first months of the relationship and thins out considerably as tenure grows. Customers who make it past the early period tend to stay, which argues for concentrating retention effort in the onboarding window rather than spreading it evenly across the base.

**The correlation heatmap confirms the numeric relationships.** Tenure correlates negatively with churn while monthly charges correlate positively, and tenure and total charges correlate strongly with each other, which is expected since total charges accumulate over time. That last relationship is worth noting because it means the two features carry overlapping information rather than independent signal.

## Phase 3: Data Preprocessing and Feature Engineering

In [ ]:
# Drop customerID as it's just an identifier
df.drop('customerID', axis=1, inplace=True)

print(f"Shape after dropping customerID: {df.shape}")

In [ ]:
# FEATURE ENGINEERING: count how many services each customer subscribes to
service_columns = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
                   'OnlineBackup', 'DeviceProtection', 'TechSupport',
                   'StreamingTV', 'StreamingMovies']

# Values indicating the customer does NOT have that service
non_service_values = ['No', 'No internet service', 'No phone service']

df['TotalServices'] = df[service_columns].apply(
    lambda row: sum(value not in non_service_values for value in row), axis=1)

print("Distribution of TotalServices:")
print(df['TotalServices'].value_counts().sort_index())
print(f"\nMean services per customer: {df['TotalServices'].mean():.2f}")
print("\nChurn rate by number of services:")
print(df.groupby('TotalServices')['Churn'].apply(lambda s: (s == 'Yes').mean() * 100).round(2))

### The Engineered Feature: TotalServices

`TotalServices` counts how many of the nine available services each customer subscribes to, ranging from phone and internet through security, backup, device protection, tech support, and streaming.

The reasoning is that bundling changes the nature of the relationship. A customer with one service is buying a commodity and can switch providers with little disruption. A customer with six services has woven the provider into their household, and leaving means replacing several things at once. That switching cost is not captured by any existing column, because the individual service fields each record a separate yes or no without expressing depth of engagement.

This feature is also directly actionable in a way that many engineered features are not. If subscription depth predicts retention, the retention team has a concrete intervention available, which is to offer an additional service rather than a discount. The churn rate breakdown printed above shows whether that relationship holds in this data.

In [ ]:
# Define features (X) and target (y)
X = df.drop('Churn', axis=1)
y = df['Churn']

# Convert target to numerical format
y = y.map({'No': 0, 'Yes': 1})

print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")

In [ ]:
# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object']).columns
numerical_cols = X.select_dtypes(include=np.number).columns

print(f"\nCategorical columns: {list(categorical_cols)} ")
print(f"Numerical columns: {list(numerical_cols)} ")

In [ ]:
# Create transformers for preprocessing
numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Create preprocessor using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

print("Preprocessor constructed.")

### Preprocessing Rationale

**StandardScaler for numerical features.** Tenure runs from 1 to roughly 72 months while total charges reach into the thousands. Logistic regression with regularization penalizes coefficient magnitude, and without standardization that penalty would fall unevenly across features purely because of their units. Scaling puts every numeric feature on a common footing so the penalty reflects predictive contribution rather than measurement scale.

**One-hot encoding for categorical features.** Contract type, payment method, and internet service have no natural ordering. Assigning integers would tell the model that a two year contract is numerically twice a one year contract, which is not a meaningful statement. One-hot encoding gives each category its own binary column and avoids inventing that ranking. Setting `handle_unknown='ignore'` prevents a failure if a category appears in new data that was absent during training.

**ColumnTransformer inside a Pipeline.** Routing each column group to its own transformer and wrapping the result with the classifier means every preprocessing parameter, meaning the scaling values and the category lists, is learned from training data alone and then applied unchanged. This prevents leakage and, just as importantly for the deployment phase, means the saved model file carries its own preprocessing. New raw customer data can be passed straight in without reproducing any transformation steps by hand.

## Phase 4: Model Building and Training

In [ ]:
# Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, \
random_state=42, stratify=y)

print(f"\nTraining set size: {X_train.shape[0]} rows")
print(f"Testing set size: {X_test.shape[0]} rows")
print(f"\nChurn rate in training set: {y_train.mean()*100:.2f}%")
print(f"Churn rate in test set: {y_test.mean()*100:.2f}%")

In [ ]:
# Create complete pipeline
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(solver='liblinear', random_state=42))
])

print("Pipeline created.")

### Why Logistic Regression Suits This Problem

The target has two outcomes, churned or retained, which is exactly what logistic regression is built for. Rather than predicting a class directly, it estimates the probability that a customer belongs to the churn class, and that probability is more useful to a retention team than a hard label. A team with budget to contact 500 customers can rank the base by risk score and work down the list, which a yes-or-no output would not support.

Interpretability is the stronger argument here. Each coefficient expresses how a feature shifts the log odds of churning, holding the others constant, and those coefficients can be converted into odds ratios that read as plain business statements. A retention strategy needs to know which levers to pull, and a model that reports month-to-month contracts as a specific multiplier on churn odds tells the team what to change. Gradient boosting would likely score somewhat higher on this data, but it would not hand the business a list of drivers in the same legible form.

The threshold is adjustable as well. The default cutoff of 0.5 can be lowered to catch more at-risk customers when the cost of missing one exceeds the cost of an unnecessary retention offer, which for churn is usually the case.

In [ ]:
# Hyperparameter tuning on the regularization parameter C
param_grid = {'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100]}

grid_search = GridSearchCV(
    model_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

print("Running GridSearchCV on regularization parameter C...")
grid_search.fit(X_train, y_train)

print(f"\nBest C value: {grid_search.best_params_['classifier__C']}")
print(f"Best cross-validated ROC-AUC: {grid_search.best_score_:.4f}")

# Use the best estimator as the final model
model_pipeline = grid_search.best_estimator_
print("\nFinal model trained with best parameters.")

`C` is the inverse of regularization strength, so a small value applies heavy shrinkage and produces a simpler model with smaller coefficients, while a large value lets coefficients grow and fits the training data more closely. There is no way to reason out the right value in advance, so the search tests six values across five cross-validation folds inside the training set and selects the one that generalizes best.

Scoring is set to `roc_auc` rather than accuracy on purpose. Given the class imbalance, accuracy would reward a model that leans toward predicting no churn, while AUC measures how well the model ranks customers by risk across every possible threshold. Ranking quality is what a retention team actually uses.

## Phase 5: Model Evaluation and Interpretation

In [ ]:
# Make predictions on test set
y_pred = model_pipeline.predict(X_test)
y_proba = model_pipeline.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred, target_names=['No Churn', 'Churn'])

print("\nModel Evaluation on Test Data:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("\nConfusion Matrix:")
print(conf_matrix)
print("\nClassification Report:")
print(class_report)

In [ ]:
# Compare against a baseline that always predicts "No Churn"
majority_baseline = (y_test == 0).mean()
print(f"Always-predict-No-Churn baseline accuracy: {majority_baseline:.4f}")
print(f"Model accuracy: {accuracy:.4f}")
print(f"Improvement over baseline: {(accuracy - majority_baseline)*100:.2f} percentage points")

In [ ]:
# Confusion matrix visualization
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix,
                              display_labels=['No Churn', 'Churn'])
disp.plot(cmap='Blues', ax=ax, values_format='d')
plt.title('Confusion Matrix - Customer Churn Prediction')
plt.grid(False)
plt.show()

In [ ]:
# ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, linewidth=2, label=f'Logistic Regression (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier (AUC = 0.5000)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Customer Churn Prediction')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

### Reading the Evaluation Output

**Accuracy needs its baseline.** The comparison cell above prints what a model would score by predicting no churn for every customer. The gap between that figure and the model's accuracy is the real measure of what the model contributes, and on an imbalanced target that gap is always narrower than the raw accuracy number suggests.

**Precision and recall answer different business questions.** Precision asks how many of the customers flagged as churn risks actually would have left, which governs how much retention budget gets spent on people who were never going to leave. Recall asks how many of the customers who actually left were flagged in advance, which governs how many were missed entirely. For retention, recall usually carries more weight, since an unnecessary discount costs a fraction of a lost customer.

**The confusion matrix shows where the errors land.** False negatives are churners the model missed, and those represent customers nobody attempted to save. False positives are retained customers who were flagged anyway, and those represent wasted offers. The counts are rarely balanced, and knowing which way they lean tells the team whether to adjust the decision threshold.

**AUC measures ranking rather than a single decision.** It reports how reliably the model scores an actual churner above a retained customer across all thresholds, which is exactly how a prioritized call list gets built. A value of 0.5 would mean no better than random ordering.

In [ ]:
# Extract trained components
log_reg = model_pipeline.named_steps['classifier']
fitted_preprocessor = model_pipeline.named_steps['preprocessor']

# Get feature names after one-hot encoding
ohe_feature_names = fitted_preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
feature_names = list(numerical_cols) + list(ohe_feature_names)

# Create coefficients DataFrame
coefficients = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': log_reg.coef_[0]
})

# Convert to odds ratios for business interpretation
coefficients['Odds_Ratio'] = np.exp(coefficients['Coefficient'])

# Sort by absolute coefficient value
coefficients['Abs_Coefficient'] = coefficients['Coefficient'].abs()
coefficients.sort_values(by='Abs_Coefficient', ascending=False, inplace=True)

print("\nTop 10 most impactful features on Churn:")
print(coefficients.head(10).to_string(index=False))

In [ ]:
# Visualize the top 15 coefficients
top_features = coefficients.head(15).sort_values('Coefficient')

plt.figure(figsize=(10, 8))
colors = ['steelblue' if c < 0 else 'indianred' for c in top_features['Coefficient']]
plt.barh(top_features['Feature'], top_features['Coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Top 15 Churn Drivers (red increases churn, blue reduces it)')
plt.xlabel('Coefficient (log-odds)')
plt.tight_layout()
plt.show()

### Interpreting the Coefficients in Business Terms

A positive coefficient raises the log odds of churning and a negative one lowers them. Exponentiating a coefficient converts it into an odds ratio, which reads more naturally: a value of 1.5 means the odds of churning are 50% higher when that condition holds, while 0.6 means they are 40% lower.

Two cautions apply to reading the table above. Coefficients on the numeric features are expressed in standardized units, so each describes the effect of a one standard deviation change rather than one month or one dollar. And these are associations rather than causal effects, so a feature that predicts churn is not automatically a lever that changes it. That distinction matters most for the retention recommendations, because moving a customer onto a longer contract only reduces churn if the contract is doing the work, rather than simply marking customers who intended to stay anyway.

The top five features by absolute coefficient should be identified from the printed output above and read against the exploratory findings. Contract type, tenure, internet service type, and payment method are the ones the visualizations suggest will rank highly, and consistency between the exploratory analysis and the fitted model is itself a useful check that the model learned real structure rather than noise.

## Phase 6: Model Deployment Simulation

In [ ]:
# Save the complete pipeline
model_filename = 'customer_churn_model.pkl'
joblib.dump(model_pipeline, model_filename)
print(f"\nTrained model pipeline saved as '{model_filename}'")

In [ ]:
# Load the saved model
loaded_model = joblib.load('customer_churn_model.pkl')
print("Model pipeline loaded successfully.")


def predict_churn(customer_data, model=loaded_model, threshold=0.5):
    """Predict churn for one customer.

    Args:
        customer_data: dict of raw customer attributes (same columns used in training).
        model: fitted pipeline that handles its own preprocessing.
        threshold: probability cutoff for flagging a customer as a churn risk.

    Returns:
        dict with the prediction label, churn probability, and a risk band.
    """
    customer_df = pd.DataFrame([customer_data])
    probability = model.predict_proba(customer_df)[0][1]
    prediction = 'Yes' if probability >= threshold else 'No'

    if probability >= 0.70:
        risk = 'High'
    elif probability >= 0.40:
        risk = 'Medium'
    else:
        risk = 'Low'

    return {'churn_prediction': prediction,
            'churn_probability': round(float(probability), 4),
            'risk_level': risk}

print("Prediction function defined.")

In [ ]:
# Helper to build a full customer record with sensible defaults
def build_customer(**overrides):
    base = {
        'gender': 'Male', 'SeniorCitizen': 0, 'Partner': 'No', 'Dependents': 'No',
        'tenure': 12, 'PhoneService': 'Yes', 'MultipleLines': 'No',
        'InternetService': 'DSL', 'OnlineSecurity': 'No', 'OnlineBackup': 'No',
        'DeviceProtection': 'No', 'TechSupport': 'No', 'StreamingTV': 'No',
        'StreamingMovies': 'No', 'Contract': 'One year', 'PaperlessBilling': 'No',
        'PaymentMethod': 'Mailed check', 'MonthlyCharges': 55.0, 'TotalCharges': 660.0
    }
    base.update(overrides)

    # Recompute the engineered feature so it stays consistent with training
    base['TotalServices'] = sum(
        base[c] not in ['No', 'No internet service', 'No phone service']
        for c in service_columns)
    return base


# EXAMPLE 1: High-risk profile
customer_1 = build_customer(
    tenure=1, InternetService='Fiber optic', Contract='Month-to-month',
    PaperlessBilling='Yes', PaymentMethod='Electronic check',
    MonthlyCharges=75.75, TotalCharges=75.75)

# EXAMPLE 2: Low-risk profile
customer_2 = build_customer(
    Partner='Yes', Dependents='Yes', tenure=68, MultipleLines='Yes',
    InternetService='DSL', OnlineSecurity='Yes', OnlineBackup='Yes',
    DeviceProtection='Yes', TechSupport='Yes', Contract='Two year',
    PaymentMethod='Bank transfer (automatic)',
    MonthlyCharges=84.50, TotalCharges=5746.00)

# EXAMPLE 3: Mixed profile
customer_3 = build_customer(
    tenure=20, InternetService='Fiber optic', StreamingTV='Yes',
    StreamingMovies='Yes', Contract='Month-to-month', PaperlessBilling='Yes',
    PaymentMethod='Credit card (automatic)',
    MonthlyCharges=95.20, TotalCharges=1904.00)

for label, customer in [('1 (new fiber, month-to-month)', customer_1),
                        ('2 (long tenure, two-year contract, bundled)', customer_2),
                        ('3 (mid tenure, fiber, month-to-month)', customer_3)]:
    result = predict_churn(customer)
    print(f"\nCustomer {label}")
    print(f"  Tenure: {customer['tenure']} months | Contract: {customer['Contract']} "
          f"| Services: {customer['TotalServices']}")
    print(f"  Predicted churn: {result['churn_prediction']} "
          f"| Probability: {result['churn_probability']} "
          f"| Risk: {result['risk_level']}")

### Deployment Considerations

**What the saved file contains.** The pickled object is the entire pipeline, not just the fitted classifier, so scaling parameters and one-hot category mappings travel with the model. A production system passes raw customer records straight in without reimplementing any preprocessing, which removes the most common source of training and serving mismatch, where preprocessing drifts apart between the two environments.

**How this would run in practice.** The natural deployment is a scheduled batch job rather than a live service. A nightly or weekly run scores the active customer base, writes probabilities back to the CRM, and delivers the retention team a ranked call list. A REST endpoint would suit a service representative seeing a risk score during a live call, but batch scoring covers the primary use case at far less operational cost.

**The threshold is a business decision, not a technical one.** The function accepts a threshold argument because 0.5 is only the default, not the right answer. If a retention offer costs \$50 and a lost customer costs \$500, flagging more customers is worth it even though more of those flags will be wrong, and the threshold should move down accordingly.

**Monitoring matters more than the initial accuracy.** Customer behavior shifts as competitors change pricing and as the company changes its own offers, so a model trained on last year's base will degrade. Performance needs tracking over time against outcomes that arrive weeks later, with retraining scheduled rather than waiting for someone to notice the model has stopped working. The engineered `TotalServices` feature has to be recomputed identically in production, which is why the helper above derives it from the service columns rather than accepting it as an input.

**One caution about acting on the output.** This model ranks customers by how likely they are to leave, which is not the same as ranking them by how much a retention offer would change their behavior. Some high-risk customers are leaving regardless and spending on them is wasted, while some would have stayed anyway. Measuring the actual effect of an intervention requires a holdout group that receives no offer, so the comparison is between treated and untreated customers rather than a raw retention rate.

## Summary and Business Recommendations

**Model performance.** The evaluation metrics, baseline comparison, and ROC curve above establish what the model delivers. The figure that matters for deployment is recall on the churn class, since customers the model fails to flag receive no intervention, and the gap between model accuracy and the always-predict-no-churn baseline is the honest measure of what the model adds.

**Primary churn drivers.** The exploratory analysis and the fitted coefficients agree on the main patterns. Month-to-month contracts carry substantially higher churn than one or two year agreements. Fiber optic subscribers leave more often than DSL subscribers despite paying more. Short tenure is the strongest risk marker, with churn concentrated in the earliest months. Higher monthly charges track with departure.

**Recommendations.** Concentrate retention effort in the first months of the relationship, since that is where churn concentrates and where intervention has the most room to work. Test contract conversion incentives for month-to-month customers, offering added value rather than a price cut so margin is preserved. Investigate the fiber optic result directly, because losing premium subscribers at an elevated rate is a product or pricing problem that a retention campaign papers over rather than solves. Use the model's probability output to rank the base and work down the list rather than treating every flagged customer identically.

**Limitations.** The eleven records removed during cleaning were all zero-tenure customers, the newest segment and the one most exposed to churn. Coefficients describe association rather than causation, so a predictor is not automatically a lever. And the model was trained on a single snapshot, so it captures the patterns present in that period rather than any shift that follows.